In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler


# 1. Load the  dataset
data_path = Path("monthly_records.csv")

df = pd.read_csv(
    data_path,
    parse_dates=["Entry_Date"]
)


# 2. Select the training history
# Reserve observations from 2024 for validation and testing.
cutoff_date = pd.Timestamp("2023-12-01")

historical_df = df.loc[
    df["Entry_Date"] <= cutoff_date
].copy()


# 3. Aggregate the decision criteria by part reference
mcdm_df = (
    historical_df
    .groupby("Reference", sort=True)
    .agg(
        Average_Quantity=("Quantity", "mean"),
        Volatility=("Quantity", "std"),
        Average_Unit_Margin=("Unit_Margin", "mean"),
        Total_Cost=("Total_Purchase_Cost", "sum")
    )
    .reset_index()
)

# Standard deviation is undefined when only one observation is available.
mcdm_df["Volatility"] = (
    mcdm_df["Volatility"].fillna(0)
)

criteria = [
    "Average_Quantity",
    "Volatility",
    "Average_Unit_Margin",
    "Total_Cost"
]

if len(mcdm_df) < 2:
    raise ValueError(
        "At least two parts are required."
    )

if not np.isfinite(
    mcdm_df[criteria].to_numpy(dtype=float)
).all():
    raise ValueError(
        "The decision criteria contain missing or infinite values."
    )


# 4. Apply Min-Max normalization to the [0, 1] range
scaler = MinMaxScaler()

normalized_values = scaler.fit_transform(
    mcdm_df[criteria]
)


# 5. Reverse the direction of the volatility criterion
# Lower volatility receives a higher normalized preference value.
volatility_index = criteria.index("Volatility")

normalized_values[:, volatility_index] = (
    1 - normalized_values[:, volatility_index]
)

normalized_df = pd.DataFrame(
    normalized_values,
    index=mcdm_df["Reference"],
    columns=criteria
)


# 6. Calculate the standard deviation of each normalized criterion
standard_deviations = normalized_values.std(
    axis=0,
    ddof=1
)


# 7. Calculate the Pearson correlation matrix
# This implementation also handles constant criteria.
centered_values = (
    normalized_values
    - normalized_values.mean(axis=0)
)

column_norms = np.linalg.norm(
    centered_values,
    axis=0
)

denominator = np.outer(
    column_norms,
    column_norms
)

correlation_matrix = np.divide(
    centered_values.T @ centered_values,
    denominator,
    out=np.zeros((len(criteria), len(criteria))),
    where=denominator > 1e-12
)

np.fill_diagonal(
    correlation_matrix,
    1.0
)

correlation_df = pd.DataFrame(
    correlation_matrix,
    index=criteria,
    columns=criteria
)


# 8. Calculate the CRITIC information scores
# Information = standard deviation × sum of (1 - correlation).
conflict_scores = (
    1 - correlation_matrix
).sum(axis=1)

information_scores = (
    standard_deviations * conflict_scores
)


# 9. Normalize the information scores to obtain criterion weights
total_information = information_scores.sum()

if total_information > 1e-12:
    weights = (
        information_scores / total_information
    )

else:
    # Use equal weights if no criterion provides discrimination.
    weights = (
        np.ones(len(criteria)) / len(criteria)
    )

weights_df = pd.DataFrame({
    "Criterion": criteria,
    "Standard_Deviation": standard_deviations,
    "Conflict": conflict_scores,
    "Information": information_scores,
    "Weight": weights
})


# 10. Check the calculated weights and normalized values
assert np.isclose(weights.sum(), 1.0)

assert (weights >= 0).all()

assert (
    (normalized_values >= 0)
    & (normalized_values <= 1)
).all()


# 11. Display the results
print("Aggregated decision criteria:")
print(mcdm_df.head())

print("\nNormalized and preference-oriented criteria:")
print(normalized_df.head())

print("\nCorrelation matrix:")
print(correlation_df.round(4))

print("\nCRITIC weights:")
print(weights_df.round(6).to_string(index=False))

print("\nSum of weights:", weights.sum())


# 12. Save the intermediate results
output_folder = Path("results/step_02")

output_folder.mkdir(
    parents=True,
    exist_ok=True
)

mcdm_df.to_csv(
    output_folder / "aggregated_criteria.csv",
    index=False
)

normalized_df.to_csv(
    output_folder / "normalized_criteria.csv",
    index=True
)

correlation_df.to_csv(
    output_folder / "criteria_correlations.csv",
    index=True
)

weights_df.to_csv(
    output_folder / "critic_weights.csv",
    index=False
)